# Lab 3: Decision Rules on Streaming Data

## Goal

- Define business rules for fraud detection,
- Apply rules to live Kafka transactions (Python consumer),
- Apply rules inside Spark Structured Streaming,
- Write alerts to a separate Kafka topic.

**Business case (continued):** Your monitoring system now needs to flag suspicious transactions based on business rules before you have a trained ML model.

---

## Part 1: Rule-Based Scoring in Python

### Task 1.1 -- Define scoring rules

A transaction is suspicious if ANY of these conditions is true:

| Rule | Condition | Score |
|---|---|---|
| R1 | amount > 3000 | +3 |
| R2 | category == "electronics" and amount > 1500 | +2 |
| R3 | hour of transaction < 6 (night) | +2 |
| R4 | amount > 2x average for that store | +1 |

If total score >= 3, flag as **SUSPICIOUS**.

Write a function `score_transaction(tx, store_averages)` that returns the total score and list of triggered rules.

In [ ]:
def score_transaction(tx, store_averages):
    """Score a transaction based on business rules.
    
    Args:
        tx: dict with keys: tx_id, user_id, amount, store, category, timestamp
        store_averages: dict {store_name: average_amount}
    Returns:
        (score: int, triggered_rules: list[str])
    """
    # YOUR CODE
    pass

# Test
test_tx = {'tx_id': 'TX999', 'user_id': 'u05', 'amount': 4500.0,
           'store': 'Warsaw', 'category': 'electronics', 'timestamp': '2026-04-01T03:15:00'}
avgs = {'Warsaw': 500, 'Krakow': 400, 'Gdansk': 350, 'Wroclaw': 450}

score, rules = score_transaction(test_tx, avgs)
print(f"Score: {score}, Rules: {rules}")  # Should be >= 3

### Task 1.2 -- Scoring consumer with learning averages

Write a Kafka consumer that:
1. Reads transactions from `transactions` topic
2. Maintains running averages per store (stateful!)
3. Scores each transaction with your rules
4. Prints ALERT for suspicious ones
5. Sends suspicious transactions to a new topic `alerts`

In [ ]:
%%file scoring_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from collections import defaultdict
import json
from datetime import datetime

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='scoring-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

# Running averages
store_totals = defaultdict(float)
store_counts = defaultdict(int)

# YOUR CODE
# For each message:
# 1. Update running averages
# 2. Compute store_averages dict
# 3. Score the transaction
# 4. If score >= 3: print ALERT and send to 'alerts' topic
# 5. Every 20 messages: print summary of alerts so far

---

## Part 2: Rules in Spark Streaming

### Task 2.1 -- Read from Kafka and apply rules in Spark

Implement the same scoring logic using Spark's `withColumn` and `when`/`otherwise`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("Lab3-Rules").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# Read from Kafka (reuse schema from Lab 2)
tx_schema = StructType([
    StructField("tx_id", StringType()),
    StructField("user_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("store", StringType()),
    StructField("category", StringType()),
    StructField("timestamp", StringType()),
])

kafka_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .option("startingOffsets", "earliest")
    .load()
)

parsed = (kafka_df
    .select(from_json(col("value").cast("string"), tx_schema).alias("data"))
    .select("data.*")
    .withColumn("ts", to_timestamp("timestamp"))
)

# YOUR CODE
# Add columns:
# - rule_r1: when(col("amount") > 3000, 3).otherwise(0)
# - rule_r2: when((col("category") == "electronics") & (col("amount") > 1500), 2).otherwise(0)
# - rule_r3: when(hour(col("ts")) < 6, 2).otherwise(0)
# - total_score: rule_r1 + rule_r2 + rule_r3
# - is_suspicious: when(col("total_score") >= 3, True).otherwise(False)
#
# Filter only suspicious and display

### Task 2.2 -- Write alerts to Kafka from Spark

Send suspicious transactions to the `alerts` topic.

In [ ]:
# YOUR CODE
# suspicious = scored.filter(col("is_suspicious") == True)
# Write to Kafka:
# .select(to_json(struct("*")).alias("value"))
# .writeStream.format("kafka")...

---

## Homework

1. Add a **Rule R4** in Spark: compare each transaction amount to the windowed average (5-min tumbling window per store). Flag if amount > 2x window average. (Hint: use a self-join on windowed aggregation or a separate stream.)
2. Read from the `alerts` topic and write alerts to a CSV file.
3. Push code to your Git repository.

**Next lab:** We'll replace rules with a real ML model served via FastAPI.

In [ ]:
spark.stop()